# GUIDE Dataset — Preprocessing & Statistical Evaluation

Dataset: `microsoft/microsoft-security-incident-prediction` (GUIDE_Train.csv / GUIDE_Test.csv).

This notebook covers:
1. Loading & basic inspection
2. Missing value analysis
3. Duplicate handling
4. Type/feature preprocessing (timestamp, categoricals)
5. Statistical evaluation (describe, class balance, correlations)

Model training is intentionally **not** included — that comes in a later pass.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

matches = glob.glob('/kaggle/input/**/GUIDE_Train.csv', recursive=True)
if not matches:
    raise FileNotFoundError('GUIDE_Train.csv not found under /kaggle/input')
DATA_DIR = os.path.dirname(matches[0])
print('Using DATA_DIR =', DATA_DIR)
print(os.listdir(DATA_DIR))

## 1. Load data

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/GUIDE_Train.csv')
test = pd.read_csv(f'{DATA_DIR}/GUIDE_Test.csv')
rankings = pd.read_csv(f'{DATA_DIR}/GUIDE_Test_Queue_Rankings.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Rankings shape:', rankings.shape)
train.head()

## 2. Basic inspection

In [ ]:
print('Memory usage (MB):', train.memory_usage(deep=True).sum() / 1e6)
train.info()

## 3. Missing value analysis

In [ ]:
missing = train.isna().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]
print(missing)

plt.figure(figsize=(10, max(4, len(missing) * 0.25)))
sns.barplot(x=missing.values, y=missing.index, color='steelblue')
plt.xlabel('% missing')
plt.title('Missing values per column (Train)')
plt.tight_layout()
plt.show()

## 4. Duplicate rows

In [ ]:
dupe_count = train.duplicated().sum()
print(f'Duplicate rows: {dupe_count} ({dupe_count / len(train):.4%})')
train = train.drop_duplicates().reset_index(drop=True)
print('Shape after dropping duplicates:', train.shape)

## 5. Preprocessing
- Drop columns that are almost entirely missing (>95%)
- Parse `Timestamp` into datetime features
- Cast low-cardinality object columns to `category`

In [ ]:
high_missing_cols = missing[missing > 95].index.tolist()
print('Dropping high-missing columns:', high_missing_cols)
train = train.drop(columns=high_missing_cols)

train['Timestamp'] = pd.to_datetime(train['Timestamp'], errors='coerce')
train['Hour'] = train['Timestamp'].dt.hour
train['DayOfWeek'] = train['Timestamp'].dt.dayofweek
train['Date'] = train['Timestamp'].dt.date

object_cols = train.select_dtypes(include='object').columns
low_card_cols = [c for c in object_cols if train[c].nunique() <= 50]
for c in low_card_cols:
    train[c] = train[c].astype('category')

print('Low-cardinality columns cast to category:', low_card_cols)
train.dtypes

## 6. Statistical evaluation

In [ ]:
train.describe(include='number').T

In [ ]:
train.describe(include='category').T

### Target class balance — `IncidentGrade`

In [ ]:
target_counts = train['IncidentGrade'].value_counts(dropna=False)
target_pct = train['IncidentGrade'].value_counts(normalize=True, dropna=False) * 100
print(pd.DataFrame({'count': target_counts, 'pct': target_pct.round(2)}))

plt.figure(figsize=(6, 4))
sns.barplot(x=target_counts.index.astype(str), y=target_counts.values, color='darkorange')
plt.title('IncidentGrade class distribution')
plt.ylabel('count')
plt.tight_layout()
plt.show()

### Category / entity distributions

In [ ]:
for col in ['Category', 'EntityType', 'EvidenceRole']:
    print(f'--- {col} ---')
    print(train[col].value_counts().head(10))
    print()

### Correlation structure (numeric features)

In [ ]:
numeric_df = train.select_dtypes(include='number')
corr = numeric_df.corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation heatmap — numeric features')
plt.tight_layout()
plt.show()

### IncidentGrade vs. Category crosstab

In [ ]:
crosstab = pd.crosstab(train['Category'], train['IncidentGrade'], normalize='index') * 100
crosstab.round(1)

## 7. Summary

- Reported missing-value rates and dropped columns above the 95% missing threshold.
- Removed exact duplicate rows.
- Parsed `Timestamp` into `Hour` / `DayOfWeek` / `Date` features.
- Cast low-cardinality text columns to `category` for memory efficiency.
- Computed descriptive statistics, target class balance, category distributions, and a numeric correlation heatmap.

Next step (future task): feature engineering + model training/evaluation on `IncidentGrade`.